In [ ]:
from sympy.solvers.diophantine.diophantine import descent
!pip install optuna

# Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import math

PI = torch.pi
MIN_PLOT = -1e2
MAX_PLOT = 1e2
PLOT_GAP = 600
LEVELS = 20

initial_points = [torch.randint(-10, 10, (2,)) for _ in range(10)]

# Utils

In [ ]:

class Function:
    def __init__(self, f, gradient):
        self.ob_f = f
        self.ob_gradient = gradient

    def f(self, x, y, lib):
        return self.ob_f(x, y, lib)

    def gradient(self, vec: torch.Tensor):
        return self.ob_gradient(vec[0], vec[1])

class Plot:
    def __init__(self):
        self.x = np.linspace(MIN_PLOT,MAX_PLOT,PLOT_GAP)
        self.y = np.linspace(MIN_PLOT,MAX_PLOT,PLOT_GAP)
        self.X, self.Y = np.meshgrid(self.x, self.y)

    def make_surface(self, fun: Function):
        Z = fun.f(self.X, self.Y, np)
        fig1 = plt.figure()
        ax1 = fig1.add_subplot(111, projection='3d')
        ax1.plot_surface(self.X, self.Y, Z, cmap='viridis')
        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        ax1.set_zlabel('f(x,y)')
        plt.title("Superficie")

    def make_contour(self, fun: Function):
        Z = fun.f(self.X, self.Y, np)
        fig2 = plt.figure()
        plt.contour(self.X, self.Y, Z, levels=LEVELS, cmap='plasma')
        plt.colorbar()
        plt.title("Contour")
        plt.xlabel('x')
        plt.ylabel('y')

    def make_contour_p(self, fun: Function, x, y):
        Z = fun.f(self.X, self.Y, np)
        fig3 = plt.figure()
        plt.contour(self.X, self.Y, Z, levels=25, cmap='plasma')
        plt.colorbar()
        plt.scatter(x, y, c=[i for i in range(len(x))], cmap='inferno', s=50, label='Puntos')
        plt.title("Contour con puntos del Descenso de Gradiente")
        plt.xlabel('x')
        plt.ylabel('y')
        plt.legend()

    def make_surface_p(self, fun: Function, x, y, z):
        Z = fun.f(self.X, self.Y, np)
        fig4 = plt.figure()
        ax1 = fig4.add_subplot(111, projection='3d')
        ax1.plot_surface(self.X, self.Y, Z, cmap='viridis', alpha=0.5)
        ax1.scatter(x, y, z, c=z, cmap='coolwarm', s=60)
        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        ax1.set_zlabel('f(x,y)')
        plt.title("Superficie con punto del Descenso de Gradiente")

    def make_plot(self, x, y):
        plt.plot(x, y, 'o')
        plt.title("Contour con puntos del Descenso de Gradiente")
        plt.xlabel('x')
        plt.ylabel('y')

    @staticmethod
    def show():
        plt.show()

plot = Plot()

class Table:
    def __init__(self, data: pd.DataFrame, names):
        self.data = data
        for name in names:
            self.data[name] = []

    def add(self, run, name, value):
        if isinstance(value, float):
            value = round(value, 6)
        self.data.loc[run, name] = value

    def show(self):
        print(self.data)

# Funcion 0
No convexa, pero por rangos si, además no tiene un mínimo absoluto, solo puntos silla periódicos.

In [ ]:

def f0(x, y, lib):
    return lib.sin(x + y) + (x - y) ** 2 - 1.5 * x + 2.5 * y + 1

def gradient0(x, y):
    nx = torch.cos(x + y) + 2*(x - y) - 1.5
    ny = torch.cos(x + y) - 2*(x + y) + 2.5
    return torch.tensor([nx, ny])

func0 = Function(f0,gradient0)
plot.make_surface(func0)
plot.make_contour(func0)
plot.show()

# Funcion 1
No es convexa, si tiene punto mínimo absoluto, en (1,1) y tiene múltiples puntos silla.

In [ ]:
def gradient1(x,y):
    nx = (6*PI*torch.sin(3*PI*x)*torch.cos(3*PI*x))+(2*(x-1)*(1+(torch.sin(3*PI*y)**2)))
    ny = ((x-1)**2)*(6*PI*torch.sin(3*PI*y)*torch.cos(3*PI*y))+(2*(y-1)*(1+(torch.sin(2*PI*y)**2)))+((y-1)**2)*(4*PI*torch.sin(2*PI*x)*torch.cos(2*PI*x))
    return torch.tensor([nx, ny])

def f1(x, y, lib):
    return (lib.sin(3 * lib.pi * x) ** 2) + ((x - 1) ** 2) * (1 + (lib.sin(3 * lib.pi * y) ** 2)) + ((y - 1) ** 2) * (1 + lib.sin(2 * lib.pi * y) ** 2)

func1 = Function(f1,gradient1)
plot.make_surface(func1)
plot.make_contour(func1)
plot.show()

# Funcion 2
No es convexa, tiene 4 puntos mímimos absolutos, tiene un mínimo local

In [ ]:
def gradient2(x,y):
    nx = 2*(2*x*((x**2)+y-11)+(x+(y**2)-7))
    ny = 2*(((x**2)+y-11)+2*y*(x-(y**2)-7))
    return torch.tensor([nx, ny])

def f2(x,y,lib):
    return ((x * x) + y - 11) ** 2 + ((x + (y * y) - 7) ** 2)

func2 = Function(f2,gradient2)
plot.make_surface(func2)
plot.make_contour(func2)
plot.show()

# Descenso Gradiente

In [ ]:
class GradientDescent:
    def __init__(self, fun: Function, table : Table):
        self.T = 25
        self.fun = fun
        self.x0=torch.tensor([2, 1])
        self.table = table

    def descent(self, alpha: float, x_t):
        puntos=[]
        imagen=[]
        for i in range(self.T):
            x_t = x_t - alpha * self.fun.gradient(x_t)
            puntos.append(x_t.tolist())
            val=self.fun.f(x_t[0], x_t[1], torch)
            imagen.append(val.item())
            if torch.isnan(val) or torch.isinf(val):
                puntos.append([float("inf"), float("inf")])
                imagen.append(float("inf"))
                break
        return puntos, imagen

    def objetive(self, trial):
        alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
        _, z= self.descent(alpha, self.x0)
        return z[-1]

    def study(self):
        self.x0=torch.randint(-10, 10, (2,))
        stud = optuna.create_study(direction="minimize")
        stud.optimize(self.objetive, n_trials=50)
        print(stud.best_params)
        optuna.visualization.plot_optimization_history(stud).show()
        optuna.visualization.plot_slice(stud, params=["alpha"]).show()
        return stud.best_params

    def show_contour(self, alpha, init):
        points, images = self.descent(alpha, init)
        plot.make_contour_p(self.fun, [x[0] for x in points], [x[1] for x in points])
        plot.make_surface_p(self.fun, [x[0] for x in points], [x[1] for x in points], images)
        plot.show()

    def run(self):
        best=self.study()
        alpha=best["alpha"]
        it=0
        best_dist = [math.inf, math.inf]
        best_init = torch.tensor([math.inf, math.inf])
        for i in initial_points:
            pts, img=self.descent(alpha, i)
            idx=img.index(min(img))
            self.table.add(it, "Iteration", idx)
            self.table.add(it, "X", i[0].item())
            self.table.add(it, "Y", i[1].item())
            self.table.add(it, "Alpha", alpha)
            self.table.add(it, "Z min", min(img))
            self.table.add(it, "Z max", max(img))
            self.table.add(it, "Z end", img[-1])
            self.table.add(it, "X min", pts[idx][0])
            self.table.add(it, "Y min", pts[idx][1])
            if [min(img), idx] < best_dist:
                best_dist = [min(img), idx]
                best_init = i
            it+=1
        self.table.show()
        self.show_contour(alpha, best_init)


# RMS Prop

In [ ]:
class RMSProp:
    def __init__(self, fun: Function, table : Table):
        self.T = 25
        self.fun = fun
        self.x0=torch.tensor([2, 1])
        self.table = table

    def descent(self, x_t, rho: float, gamma: float):
        points = []
        images = []
        eps = 1e-15
        si = torch.tensor([0, 0])
        for i in range(self.T):
            grad = self.fun.gradient(x_t)
            si = si * gamma + grad * grad * (1 - gamma)
            alpha = rho / torch.sqrt(si + eps)
            x_t = x_t - alpha * grad
            points.append(x_t.tolist())
            images.append(self.fun.f(x_t[0], x_t[1], torch).item())
        return points, images

    def objetive(self, trial):
        rho = trial.suggest_float("rho", 1e-11, 1, log=True)
        gamma = trial.suggest_float("gamma", 1e-11, 1, log=True)
        _, z= self.descent(self.x0, rho, gamma)
        return z[-1]

    def study(self):
        self.x0=torch.randint(-10, 10, (2,))
        stud = optuna.create_study(direction="minimize")
        stud.optimize(self.objetive, n_trials=50)
        print(stud.best_params)
        optuna.visualization.plot_optimization_history(stud).show()
        optuna.visualization.plot_slice(stud, params=["rho", "gamma"]).show()
        return stud.best_params

    def show_contour(self,gamma, rho, init):
        pts, img = self.descent(init,rho,gamma)
        plot.make_contour_p(self.fun, [x[0] for x in pts], [x[1] for x in pts])
        plot.make_surface_p(self.fun, [x[0] for x in pts], [x[1] for x in pts], img)
        plot.show()

    def run(self):
        best = self.study()
        rho, gamma = best["rho"], best["gamma"]
        it = 0
        best_dist = [math.inf, math.inf]
        best_init = torch.tensor([math.inf, math.inf])
        for i in initial_points:
            points, images = self.descent(i, rho, gamma)
            idx=images.index(min(images))
            self.table.add(it, "X", i[0].item())
            self.table.add(it, "Y", i[1].item())
            self.table.add(it, "Rho", rho)
            self.table.add(it, "Gamma", gamma)
            self.table.add(it, "Z min", min(images))
            self.table.add(it, "Iteration", idx)
            self.table.add(it, "Z max", max(images))
            self.table.add(it, "Z end", images[-1])
            self.table.add(it, "X min", points[idx][0])
            self.table.add(it, "Y min", points[idx][1])
            if [min(images), idx] < best_dist:
                best_dist = [min(images), idx]
                best_init = i
            it += 1
        self.table.show()
        self.show_contour(gamma, rho, best_init)


# Análisis con Optuna

In [ ]:
GD0 = GradientDescent(func0, Table(pd.DataFrame(), ["X","Y","Alpha","Iteration","Z min", "Z max","Z end", "X min", "Y min"]))
GD0.run()

In [ ]:
RMS0 = RMSProp(func0, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end", "X min", "Y min"]))
RMS0.run()

In [ ]:
GD1 = GradientDescent(func1, Table(pd.DataFrame(), ["X","Y","Alpha","Iteration","Z min", "Z max","Z end"]))
GD1.run()

In [ ]:
RMS1 = RMSProp(func1, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end"]))
RMS1.run()

In [ ]:
GD2 = GradientDescent(func2, Table(pd.DataFrame(), ["X","Y","Alpha","Iteration","Z min", "Z max","Z end"]))
GD2.run()

In [ ]:
RMS2 = RMSProp(func2, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end"]))
RMS2.run()